# Practical 5b: Reading errors, debugging and testing your code

<div class="alert alert-block alert-success">
<b>Objectives:</b> By now you have written loops, functions and even solved differential equations. You will also have seen plenty of red error messages along the way! In this practical we take a step back and learn how to deal with code that does not work, and how to check that code which <i>appears</i> to work is actually giving the right answer. We will do this through 3 sections:

 - 1) [Reading error messages (tracebacks)](#Part1)
      * [Exercise 1: Error detective - fix four broken Earth-science snippets](#Exercise1)
 - 2) [Debugging code that runs but gives the wrong answer](#Part2)
      * [Exercise 2: Debug a radiocarbon dating function](#Exercise2)
 - 3) [Writing simple tests with <code>assert</code>](#Part3)
      * [Exercise 3: Write tests for a saturation vapour pressure function](#Exercise3)
      * [Exercise 4: Use tests to find and fix the bugs in a dry-spell counter](#Exercise4)
      * [Exercise 5 (extension): Make a function reject impossible inputs](#Exercise5)

As with our other notebooks, please note that you should not feel pressured to complete every exercise in class. These practicals are designed for you to take outside of class and continue working on them. Proposed solutions to all exercises can be found in the 'Solutions' folder.
</div>

Professional scientists and software developers spend a large fraction of their time debugging. Getting an error message is not a sign that you are bad at programming; it is a normal part of writing code. The skills in this practical will save you hours in the rest of the course and in your assessment.

<div class="alert alert-block alert-warning">
<b>Please note:</b> After reading the instructions and aims of any exercise, search the code snippets for a note that reads -------'INSERT CODE HERE'------- to identify where you need to write your code.

Some code boxes in this notebook are <b>meant</b> to produce an error when you first run them. That is the point! Read the message carefully before you change anything.
</div>

## 1) Reading error messages (tracebacks) <a name="Part1"></a>

When Python cannot run your code it stops and prints a **traceback**. A traceback can look intimidating, but it always contains the same three pieces of information:

 1. **Where** the problem happened: the file (or notebook cell) and the line number, marked with an arrow <code>----></code> or with <code>^^^^</code> underneath the offending code.
 2. **What kind** of problem it is: the *error type*, such as <code>NameError</code> or <code>TypeError</code>.
 3. **Why**: a short message after the error type, such as <code>name 'rainfal' is not defined</code>.

<b>The golden rule: read a traceback from the bottom up.</b> The last line tells you what went wrong. The lines above show the chain of function calls that led there, with the most recent call at the bottom.

Let's look at an example. The function below converts a temperature from Kelvin to degrees Celsius. Run the code box and look at the traceback.

In [1]:
# A function that converts temperature from Kelvin to degrees Celsius
def kelvin_to_celsius(temperature_K):
    temperature_C = temperature_K - 273.15
    return temperature_C

# Imagine we have read this value from a text file of weather station data.
# Values read from text files often arrive as strings, not numbers!
station_temperature = "288.4"

print(kelvin_to_celsius(station_temperature))

TypeError: unsupported operand type(s) for -: 'str' and 'float'

Reading this from the bottom up:

 - The last line says <code>TypeError: unsupported operand type(s) for -: 'str' and 'float'</code>. So we tried to subtract (<code>-</code>) a <code>float</code> from a <code>str</code> (a string of text). Python does not know how to subtract a number from a word.
 - The line above shows the arrow pointing at <code>temperature_C = temperature_K - 273.15</code>, **inside** our function. That is where Python gave up.
 - Higher up, the traceback shows the line <code>print(kelvin_to_celsius(station_temperature))</code>. That is the call that sent the bad value *into* the function.

Notice that the line where the error *appears* (inside the function) is not where the *cause* is. The real problem is that <code>station_temperature</code> is the string <code>"288.4"</code> rather than the number <code>288.4</code>. We can fix it by converting it with <code>float()</code>:

In [2]:
def kelvin_to_celsius(temperature_K):
    temperature_C = temperature_K - 273.15
    return temperature_C

station_temperature = "288.4"
station_temperature = float(station_temperature) # convert the string to a floating point number

print(kelvin_to_celsius(station_temperature))

15.25


### A field guide to common errors

You will meet the following error types again and again. It is worth getting to know them:

| Error type | What it usually means | Earth-science flavoured example |
|---|---|---|
| <code>SyntaxError</code> | Python cannot even read your code: a missing colon, bracket or quote | <code>for year in years</code> (no <code>:</code> at the end) |
| <code>IndentationError</code> | The spaces at the start of a line are wrong | The body of a <code>for</code> loop is not indented |
| <code>NameError</code> | You used a variable or function name that does not exist (often a typo, or a cell you have not run yet) | <code>print(rainfal)</code> instead of <code>print(rainfall)</code> |
| <code>TypeError</code> | You used a value of the wrong *type*, e.g. text where a number is needed | <code>"1085" * 0.0065</code> |
| <code>IndexError</code> | You asked for a position in a list or array that does not exist | Asking for <code>magnitudes[10]</code> in a list of 10 earthquakes |
| <code>KeyError</code> | You asked a dictionary for a key it does not have (keys are case sensitive!) | <code>vei['etna']</code> when the key is <code>'Etna'</code> |
| <code>ValueError</code> | The type is right but the value is not allowed | <code>float("12.3 mm")</code> |
| <code>ZeroDivisionError</code> | You divided by zero | Mean rainfall over a period with 0 days |

A <code>SyntaxError</code> is a little different from the others: Python finds it *before* running anything, so none of the code in that box will run. Try it below. Can you spot the problem from the <code>^</code> marker?

In [3]:
eruption_years = [1669, 1928, 1981, 2002]

for year in eruption_years
    print("Etna erupted in", year)

SyntaxError: expected ':' (1851496475.py, line 3)

<div class="alert alert-block alert-info">
<b>Tip: using GenAI to explain an error.</b> If a traceback still does not make sense, you can paste it (along with the code that produced it) into a GenAI assistant and ask it to <i>explain</i> the error. See the <code>genai_python_guide.ipynb</code> notebook for advice. However, always read the traceback yourself first, and check that any suggested fix makes sense: the assistant cannot see your data or the rest of your notebook, and it will sometimes be confidently wrong.
</div>

<div class="alert alert-block alert-success">
<b> Exercise 1: Error detective </b>   <a name="Exercise1"></a>

Each of the four code boxes below contains a short piece of Earth-science code with <b>one</b> mistake in it. For each box:

 - Run it and read the traceback from the bottom up
 - Fill in the comment lines at the top of the box: what is the <b>error type</b>, which <b>line</b> caused it, and <b>why</b> did it happen?
 - Fix the code so that it runs and prints a sensible answer

The expected answers are given in each box so you know when you have succeeded.
</div>

In [4]:
# (a) Mean monthly rainfall in Manchester [mm] (approximate long-term values)
#-------'INSERT CODE HERE'-------
# Error type:
# Line:
# Why:
#--------------------------------
rainfall = [79, 59, 63, 52, 58, 69, 66, 79, 75, 97, 92, 95]

total = 0
for month in rainfall:
    total = total + month

mean_rainfall = total / len(rainfal)
print("Mean monthly rainfall =", round(mean_rainfall, 1), "mm") # Expected answer: 73.7 mm

NameError: name 'rainfal' is not defined

In [5]:
# (b) Air pressure at the summit of Snowdon using the International Standard Atmosphere
# p = 101325 * (1 - 2.25577e-5 * h)**5.25588, with h in metres and p in Pascals
#-------'INSERT CODE HERE'-------
# Error type:
# Line:
# Why:
#--------------------------------
summit_height = "1085" # metres, as read from a text file

pressure = 101325 * (1 - 2.25577e-5 * summit_height)**5.25588
print("Pressure at the summit =", round(pressure / 100, 1), "hPa") # Expected answer: 889.5 hPa

TypeError: can't multiply sequence by non-int of type 'float'

In [6]:
# (c) Print the magnitude of each earthquake in a list, one per line
#-------'INSERT CODE HERE'-------
# Error type:
# Line:
# Why:
#--------------------------------
magnitudes = [4.2, 5.1, 3.8, 6.4, 4.9]

for i in range(1, len(magnitudes) + 1):
    print("Earthquake", i, "had magnitude", magnitudes[i])
# Expected answer: five lines, starting with 'Earthquake 1 had magnitude 4.2'

Earthquake 1 had magnitude 5.1
Earthquake 2 had magnitude 3.8
Earthquake 3 had magnitude 6.4
Earthquake 4 had magnitude 4.9


IndexError: list index out of range

In [7]:
# (d) Look up the Volcanic Explosivity Index (VEI) of a famous eruption
#-------'INSERT CODE HERE'-------
# Error type:
# Line:
# Why:
#--------------------------------
vei = {'Vesuvius 79 AD': 5, 'Krakatau 1883': 6, 'Pinatubo 1991': 6, 'Etna 2002': 2}

eruption = 'pinatubo 1991'
print("The VEI of", eruption, "was", vei[eruption]) # Expected answer: 6

KeyError: 'pinatubo 1991'

## 2) Debugging code that runs but gives the wrong answer <a name="Part2"></a>

Error messages are actually the *friendly* kind of bug: Python tells you something is wrong and where. The more dangerous kind is a **logic error**, where the code runs happily and gives you a number, but the number is wrong. Nothing turns red. If you do not check, that wrong number could end up in your report.

Here is a toolkit of debugging strategies that work for any language:

 1. **Check against an answer you already know.** Before trusting a function, give it an input where you can work out the answer by hand, or where the answer is well known.
 2. **Print the intermediate values.** Add <code>print()</code> statements inside your code so you can see what each variable holds at each step. An f-string makes this quick: <code>print(f"i={i}, total={total}")</code>.
 3. **Make the problem smaller.** Try your code on 3 values rather than 3,000, so you can check every step by hand.
 4. **Change one thing at a time,** then re-run. If you change five things at once you will not know which one mattered.
 5. **Explain your code out loud,** line by line, to a friend (or a rubber duck!). You will often hear yourself say something that is not what the code actually does.

As an example, the code below is supposed to calculate the total annual rainfall in Manchester from our monthly values. It runs without any error. Is the answer right?

In [8]:
rainfall = [79, 59, 63, 52, 58, 69, 66, 79, 75, 97, 92, 95] # Jan to Dec [mm]

total = 0
for i in range(1, 12):
    total = total + rainfall[i]

print("Total annual rainfall =", total, "mm")

Total annual rainfall = 805 mm


Strategy 1: we can check this a different way, using Python's built-in <code>sum()</code> function, which gives 884 mm. So the loop is wrong by 79 mm. That number looks familiar... it is January's rainfall!

Strategy 2: let's add a <code>print()</code> statement inside the loop so we can see exactly which months are being added:

In [9]:
rainfall = [79, 59, 63, 52, 58, 69, 66, 79, 75, 97, 92, 95] # Jan to Dec [mm]

print("Check using sum():", sum(rainfall), "mm")

total = 0
for i in range(1, 12):
    total = total + rainfall[i]
    print(f"i = {i}, adding {rainfall[i]} mm, running total = {total} mm") # debugging print statement

Check using sum(): 884 mm
i = 1, adding 59 mm, running total = 59 mm
i = 2, adding 63 mm, running total = 122 mm
i = 3, adding 52 mm, running total = 174 mm
i = 4, adding 58 mm, running total = 232 mm
i = 5, adding 69 mm, running total = 301 mm
i = 6, adding 66 mm, running total = 367 mm
i = 7, adding 79 mm, running total = 446 mm
i = 8, adding 75 mm, running total = 521 mm
i = 9, adding 97 mm, running total = 618 mm
i = 10, adding 92 mm, running total = 710 mm
i = 11, adding 95 mm, running total = 805 mm


The printout shows that <code>i</code> starts at 1, not 0, so January (position 0) is never added. The fix is to use <code>range(12)</code>, or even better <code>range(len(rainfall))</code>, which still works if the list changes length. Once the bug is fixed, remember to remove (or comment out) the debugging <code>print()</code> statements so they do not clutter your output:

In [10]:
rainfall = [79, 59, 63, 52, 58, 69, 66, 79, 75, 97, 92, 95] # Jan to Dec [mm]

total = 0
for i in range(len(rainfall)): # i now runs from 0 (January) to 11 (December)
    total = total + rainfall[i]
    # print(f"i = {i}, adding {rainfall[i]} mm, running total = {total} mm") # debugging print, now commented out

print("Total annual rainfall =", total, "mm")
print("Check using sum():", sum(rainfall), "mm")

Total annual rainfall = 884 mm
Check using sum(): 884 mm


<div class="alert alert-block alert-info">
<b>Going further: the <code>%debug</code> command.</b> When a cell has just stopped with an error, you can run <code>%debug</code> in a new cell. This opens an interactive debugger at the point where the code crashed, and you can type the name of any variable to see its value (type <code>q</code> to quit). You do not need it for this course, but it is a powerful tool once you are comfortable with printing.
</div>

<div class="alert alert-block alert-success">
<b> Exercise 2: Debug a radiocarbon dating function </b>   <a name="Exercise2"></a>

In Practical 5 we solved the radioactive decay equation $\frac{dN}{dt} = -\lambda N$ numerically. This equation also has an exact solution, $N(t) = N_0 e^{-\lambda t}$, which is the basis of radiocarbon dating. If we measure the fraction of carbon-14 remaining in a sample, $F = N/N_0$, we can rearrange to find the age of the sample:

\begin{equation*}
t = \frac{-\ln(F)}{\lambda}, \qquad \lambda = \frac{\ln(2)}{t_{1/2}}
\end{equation*}

where $t_{1/2} = 5730$ years is the half-life of carbon-14 and $\ln$ is the natural logarithm.

A colleague has written the function below, but it gives nonsense. It contains <b>three</b> bugs. Your task is to:

 - Use strategy 1: think of some fractions where you <i>know</i> the answer. For example, a sample with half its carbon-14 left ($F = 0.5$) must be exactly one half-life old. What about $F = 0.25$? What about $F = 1$?
 - Use strategy 2: add <code>print()</code> statements inside the function to check the value of <code>decay_constant</code> and of the logarithm. Does each one have the value you would expect?
 - Fix all three bugs. Hint: check the formula above very carefully, symbol by symbol, against the code. Also check which NumPy function gives a <i>natural</i> logarithm.

When your function is correct, the check box below it will print a message saying so.
</div>

In [11]:
import numpy as np

def radiocarbon_age(fraction_remaining):
    # Calculate the age of a sample [years] from the fraction of carbon-14 remaining
    half_life = 5730.0 # half-life of carbon-14 [years]
    #-------'INSERT CODE HERE'-------
    # Add print() statements to investigate, then fix the three bugs
    decay_constant = half_life / np.log(2)
    age = np.log10(fraction_remaining) / decay_constant
    #--------------------------------
    return age

# Strategy 1: try some fractions where we know the answer
print("F = 0.5  gives", radiocarbon_age(0.5), "years (should be 5730)")
print("F = 0.25 gives", radiocarbon_age(0.25), "years (should be 11460, two half-lives)")
print("F = 1.0  gives", radiocarbon_age(1.0), "years (should be 0, a brand new sample)")

F = 0.5  gives -3.64150249142166e-05 years (should be 5730)
F = 0.25 gives -7.28300498284332e-05 years (should be 11460, two half-lives)
F = 1.0  gives 0.0 years (should be 0, a brand new sample)


In [12]:
# Run this box to check your function. If it stops with an AssertionError, there is still a bug!
# (We will learn exactly how 'assert' works in the next section.)
assert np.isclose(radiocarbon_age(0.5), 5730.0), "One half-life should be 5730 years"
assert np.isclose(radiocarbon_age(0.25), 11460.0), "Two half-lives should be 11460 years"
assert np.isclose(radiocarbon_age(1.0), 0.0), "A sample with all of its carbon-14 should be 0 years old"
print("Well done, your radiocarbon_age function passes all the checks!")
print("A sample with 30% of its carbon-14 remaining is", round(radiocarbon_age(0.3)), "years old")

AssertionError: One half-life should be 5730 years

## 3) Writing simple tests with <code>assert</code> <a name="Part3"></a>

In Exercise 2 we checked our function by printing results and comparing them with known answers by eye. That works, but it is easy to forget to do it again after you change your code. A **test** is a small piece of code that does the checking *for* you.

The simplest tool for this in Python is the <code>assert</code> statement:

<code>assert condition, "message to show if the condition is False"</code>

If the condition is <code>True</code>, nothing happens and the code carries on. If it is <code>False</code>, Python stops with an <code>AssertionError</code> and shows your message. Run the box below to see both cases.

In [13]:
number_of_months = 12

assert number_of_months == 12, "A year should have 12 months" # True, so nothing happens
print("The first assert passed")

assert number_of_months == 13, "A year should have 13 months" # False, so Python stops here
print("You will never see this line")

The first assert passed


AssertionError: A year should have 13 months

### A warning about comparing decimal numbers

Computers store decimal numbers (floats) with a tiny amount of rounding error. This means that a test using <code>==</code> on floats can fail even when your code is correct. Run the box below. Are you surprised?

In [14]:
import numpy as np

print(0.1 + 0.2)
print(0.1 + 0.2 == 0.3)            # Exact comparison: False because of rounding error!
print(np.isclose(0.1 + 0.2, 0.3))  # 'Close enough' comparison: True

# np.isclose also lets you say how close is close enough, using an absolute tolerance 'atol'
measured_temperature = 15.03
print(np.isclose(measured_temperature, 15.0, atol=0.05)) # True: within 0.05 of 15.0

0.30000000000000004
False
True
True


**Rule of thumb:** use <code>==</code> to test whole numbers, strings and True/False values. Use <code>np.isclose()</code> to test floats.

### What makes a good test?

Good tests check a mixture of the following:

 - **Known values:** an input where you know the right answer from a textbook, a hand calculation or a physical law. For example, water boils at 100 °C at sea level.
 - **Physical behaviour:** properties your answer must have, even if you do not know the exact number. For example, *air pressure must decrease with height* or *a concentration can never be negative*.
 - **Edge cases:** the unusual inputs at the boundaries, where bugs like to hide. For example an empty list, a list with one value, all zeros, a value exactly on a threshold, or something at the very start or very end of a dataset.

It is common practice to group tests into a function whose name starts with <code>test_</code>. Then you can re-run every test with a single line whenever you change your code. Here is an example for our <code>kelvin_to_celsius</code> function from Part 1:

In [15]:
def kelvin_to_celsius(temperature_K):
    return temperature_K - 273.15

def test_kelvin_to_celsius():
    # Known values
    assert np.isclose(kelvin_to_celsius(273.15), 0.0), "Water freezes at 273.15 K, which is 0 C"
    assert np.isclose(kelvin_to_celsius(373.15), 100.0), "Water boils at 373.15 K, which is 100 C"
    # Edge case: absolute zero
    assert np.isclose(kelvin_to_celsius(0.0), -273.15), "Absolute zero is -273.15 C"
    # Physical behaviour: a warmer temperature in K must give a warmer temperature in C
    assert kelvin_to_celsius(300.0) > kelvin_to_celsius(290.0), "Conversion should preserve order"
    print("All kelvin_to_celsius tests passed!")

test_kelvin_to_celsius()

All kelvin_to_celsius tests passed!


<div class="alert alert-block alert-success">
<b> Exercise 3: Write tests for a saturation vapour pressure function </b>   <a name="Exercise3"></a>

The saturation vapour pressure, $e_s$, is the pressure of water vapour in air that is saturated (100% relative humidity). It is central to cloud formation, humidity and evaporation. A widely used approximation is the Tetens equation:

\begin{equation*}
e_s(T) = 0.6108 \exp\left(\frac{17.27\, T}{T + 237.3}\right)
\end{equation*}

where $T$ is temperature in °C and $e_s$ is in kPa. The function is written for you below. Your task is to complete <code>test_saturation_vapour_pressure()</code> with <b>at least four</b> <code>assert</code> statements. Use the following facts:

 - <b>Known value:</b> at exactly 0 °C, the exponent is zero, so $e_s$ should be exactly 0.6108 kPa
 - <b>Known value:</b> from meteorological tables, at 20 °C, $e_s$ is about 2.34 kPa (use <code>np.isclose</code> with <code>atol=0.01</code>)
 - <b>Physical behaviour:</b> warmer air can hold more water vapour, so $e_s$ at 25 °C should be larger than at 15 °C
 - <b>Physical behaviour:</b> a pressure can never be negative, even in very cold air such as -40 °C

Once your tests pass, try to "break" the function on purpose, for example by changing 17.27 to 1.727, and re-run the box. Do your tests catch the mistake? A good test should fail when the code is wrong! Remember to change it back afterwards.
</div>

In [16]:
import numpy as np

def saturation_vapour_pressure(temperature_C):
    # Tetens equation: saturation vapour pressure [kPa] from temperature [degrees C]
    return 0.6108 * np.exp(17.27 * temperature_C / (temperature_C + 237.3))

def test_saturation_vapour_pressure():
    #-------'INSERT CODE HERE'-------
    # Write at least four assert statements, one for each fact in the exercise description
    # For example: assert np.isclose(saturation_vapour_pressure(0.0), ...), "..."

    #--------------------------------
    print("All saturation_vapour_pressure tests passed!")

test_saturation_vapour_pressure()

All saturation_vapour_pressure tests passed!


<div class="alert alert-block alert-success">
<b> Exercise 4: Use tests to find and fix the bugs in a dry-spell counter </b>   <a name="Exercise4"></a>

The length of the longest run of consecutive dry days is an important statistic for drought monitoring and water resource planning. In this exercise, we count a day as <b>dry</b> if it had <b>less than 1.0 mm</b> of rain.

The function <code>longest_dry_spell</code> below takes a list of daily rainfall totals and should return the length of the longest run of dry days. For example, for the rainfall record <code>[5.0, 0.0, 0.2, 0.0, 3.1, 0.0, 0.0, 2.4]</code> the answer should be 3 (days 2, 3 and 4).

The function runs without any error messages, but it contains <b>two</b> bugs. The test function below it currently contains only one test, which passes. Your tasks are:

 - Add more tests to <code>test_longest_dry_spell()</code>. Include the example above, plus edge cases. Ideas: a record that is entirely dry, an empty list, a record where the longest dry spell is at the very <i>end</i>, and a day with exactly 1.0 mm of rain (which should count as wet).
 - Run the tests. When one fails, use the strategies from Part 2 (printing inside the loop, working through a short example by hand) to work out why.
 - Fix the function, then re-run <b>all</b> of your tests until they all pass.
 - Finally, run the last code box to find the longest dry spell in a month of (made-up) rainfall data. You should find it is 9 days.

<b>Hint:</b> think about what should happen to the <code>current</code> counter on a wet day, and whether <code>longest</code> is checked after the very last day.
</div>

In [17]:
def longest_dry_spell(daily_rain, threshold=1.0):
    # Return the length of the longest run of consecutive days with rain < threshold [mm]
    #-------'INSERT CODE HERE'-------
    # This function contains two bugs. Find them with your tests, then fix them.
    longest = 0 # the longest dry spell found so far
    current = 0 # the length of the dry spell we are currently in
    for rain in daily_rain:
        if rain < threshold:
            current = current + 1
        else:
            if current > longest:
                longest = current
    #--------------------------------
    return longest

def test_longest_dry_spell():
    assert longest_dry_spell([2.0, 3.0, 4.0]) == 0, "A record with no dry days should give 0"
    #-------'INSERT CODE HERE'-------
    # Add more tests here, including the edge cases suggested in the exercise description

    #--------------------------------
    print("All longest_dry_spell tests passed!")

test_longest_dry_spell()

All longest_dry_spell tests passed!


In [18]:
# A month of made-up daily rainfall totals [mm] for a weather station
november_rain = [4.2, 0.0, 0.0, 0.4, 7.8, 12.1, 0.0, 0.0, 0.0, 0.0,
                 0.0, 0.0, 0.2, 0.0, 0.0, 3.3, 1.0, 0.0, 0.6, 9.4,
                 15.2, 2.8, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.8]

print("Longest dry spell in November:", longest_dry_spell(november_rain), "days")

Longest dry spell in November: 14 days


<div class="alert alert-block alert-success">
<b> Exercise 5 (extension): Make a function reject impossible inputs </b>   <a name="Exercise5"></a>

Tests check that our code gives the right answer for sensible inputs. But what happens if someone gives our <code>radiocarbon_age</code> function a value that is physically impossible? A fraction of carbon-14 remaining can only be between 0 and 1. Try <code>radiocarbon_age(1.5)</code> or <code>radiocarbon_age(-0.2)</code> with your fixed function from Exercise 2: you will get a negative age, or a <code>nan</code> (not a number) with a warning. Neither is helpful, and a <code>nan</code> can quietly spread through the rest of your calculations.

It is much better for the function to stop with a clear error message. We can do this ourselves with <code>raise</code>:

<code>raise ValueError("a helpful message explaining what was wrong")</code>

Your tasks are:

 - Complete the new version of <code>radiocarbon_age</code> below so that it raises a <code>ValueError</code> if <code>fraction_remaining</code> is less than or equal to 0, or greater than 1
 - Run the test box underneath. It uses <code>try</code> and <code>except</code> to check that a <code>ValueError</code> really is raised. Read it carefully to understand how it works

Notice that you have now written your own error message, just like the ones you learned to read in Part 1!
</div>

In [19]:
import numpy as np

def radiocarbon_age(fraction_remaining):
    # Calculate the age of a sample [years] from the fraction of carbon-14 remaining
    #-------'INSERT CODE HERE'-------
    # Check that fraction_remaining is greater than 0 and no more than 1. If not, raise a ValueError

    #--------------------------------
    half_life = 5730.0 # half-life of carbon-14 [years]
    decay_constant = np.log(2) / half_life
    age = -np.log(fraction_remaining) / decay_constant
    return age

In [20]:
def test_radiocarbon_age_rejects_bad_input():
    # The function should still give the right answer for a sensible input
    assert np.isclose(radiocarbon_age(0.5), 5730.0), "One half-life should be 5730 years"

    for bad_fraction in [1.5, 0.0, -0.2]:
        try:
            radiocarbon_age(bad_fraction) # This line should raise a ValueError...
        except ValueError:
            print(f"Good: radiocarbon_age({bad_fraction}) raised a ValueError") # ...and if it does, we end up here
        else:
            # The 'else' part only runs if NO error was raised, which means the test has failed
            raise AssertionError(f"radiocarbon_age({bad_fraction}) should have raised a ValueError")
    print("All input checks passed!")

test_radiocarbon_age_rejects_bad_input()

AssertionError: radiocarbon_age(1.5) should have raised a ValueError

<div class="alert alert-block alert-info">
<b>Going further: testing beyond notebooks.</b> In larger projects, tests are kept in separate files with names like <code>test_rainfall.py</code>, and a tool called <a href="https://docs.pytest.org">pytest</a> finds and runs every function whose name starts with <code>test_</code> automatically. The <code>assert</code> statements you have written today work in exactly the same way there, so you have already learned the core skill. Many research groups run their tests automatically every time they change their code, so that a bug introduced today is caught today rather than after a paper is published.
</div>

## ⚙️ What We Did

> 🟦 **In this practical, we:**
>
> - Learned to **read a traceback from the bottom up**: the error type, the message and the line that caused it
> - Met the most **common error types**: `SyntaxError`, `NameError`, `TypeError`, `IndexError`, `KeyError`, `ValueError` and friends
>
> - **Exercise 1:** Played **error detective**, diagnosing and fixing four broken Earth-science snippets
>
> - Built a **debugging toolkit** for code that runs but gives the wrong answer: check against known answers, print intermediate values, make the problem smaller, change one thing at a time, explain your code out loud
>
> - **Exercise 2:** Debugged a **radiocarbon dating** function using known half-life answers and `print()` statements
>
> - Wrote **tests with `assert`**, using `np.isclose()` to compare floats safely
>
> - **Exercise 3:** Tested the **Tetens saturation vapour pressure** equation against known values and physical behaviour
> - **Exercise 4:** Used **edge-case tests** to expose and fix two bugs in a **dry-spell counter**
> - **Exercise 5 (extension):** Made a function **raise a `ValueError`** for impossible inputs, and tested it with `try` / `except`
>
> 🧠 *Big takeaway: code that runs is not the same as code that is right. Read your errors, check your answers against things you know, and let tests do the checking for you.*